In [0]:
from pyspark.sql import functions as F, Row, Window
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType, IntegerType
import hashlib

catalog = "cinedata"
silver_schema_name = "silver"
gold_schema_name = "gold"

silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

def generate_sk(*values):
    """
    Gera Surrogate Key (SK) de 64 bits a partir de valores concatenados.
    
    Motivo de usar SK:
    - Estabilidade: IDs naturais podem mudar (TMDB refatorado? Muda tudo)
    - Performance: BIGINT é mais rápido que STRING em joins
    - Normalização: Dimensões obtêm ID único mesmo se valor repete
    - Rastreabilidade: SK permite auditoria (qual source gerou?)
    """
    combined = "|".join(str(v) if v is not None else "NULL" for v in values)
    hash_obj = hashlib.sha256(combined.encode())
    return int.from_bytes(hash_obj.digest()[:8], byteorder='big') & 0x7FFFFFFFFFFFFFFF

generate_sk_udf = F.udf(generate_sk, LongType())

print("Setup completo: Funções de Surrogate Key definidas")


Setup completo: Funções de Surrogate Key definidas


---

## BLOCO 1: gold.dim_movies

**Origem:** silver.tb_info_filmes

### Regras de Negócio Aplicadas:

#### REGRA 1: Gerar Surrogate Key para dimensão
`Motivo: SK garante identificação estável, mesmo se id_filme mudar. Usar id_filme como base para hash`

#### REGRA 2: Manter chave natural para auditoria
`Motivo: Rastreabilidade; permitir validação cruzada com Bronze`

#### REGRA 3: Remover colunas não essenciais
`Motivo: Dimensão deve ter apenas atributos (não métricas)`

***

In [0]:
print("\n[1/8] Criando: gold.dim_movies\n")

df_silver_info = spark.table(f"{silver_schema}.tb_info_filmes")

df_dim_movies = (
    df_silver_info
    .withColumn("sk_movie_id", generate_sk_udf(F.col("id_filme")))
    .select(
        "sk_movie_id",
        "id_filme",
        "titulo",
        "titulo_original",
        "data_lancamento",
        "ano_lancamento",
        "duracao_minutos",
        "idioma_original",
        "status_filme",
        "sinopse"
    )
)

print("Quality Checks para gold.dim_movies:\n")
print(f"  dim_movies: {df_dim_movies.count()} registros")
print(f"  sk_movie_id único: {df_dim_movies.select('sk_movie_id').distinct().count() == df_dim_movies.count()}")

df_dim_movies.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.dim_movies")

print(f"\ngold.dim_movies gravada")



[1/8] Criando: gold.dim_movies

Quality Checks para gold.dim_movies:

  dim_movies: 97879 registros
  sk_movie_id único: True

gold.dim_movies gravada


---

## BLOCO 2: gold.dim_genres

**Origem:** silver.tb_generos

### Regras de Negócio Aplicadas:

#### REGRA 1: Extrair gêneros únicos
`Motivo: Dimensão contém CADA gênero uma única vez, sem redundância`

#### REGRA 2: Gerar SK para cada gênero
`Motivo: Identificador estável para relacionamentos M:N na fact table`

#### REGRA 3: Ordenar para determinismo
`Motivo: Resultados consistentes entre execuções`

***

In [0]:
print("\n[2/8] Criando: gold.dim_genres\n")

df_silver_genres = spark.table(f"{silver_schema}.tb_generos")

df_dim_genres = (
    df_silver_genres
    .select("genero")
    .distinct()
    .withColumn("sk_genre_id", generate_sk_udf(F.col("genero")))
    .withColumnRenamed("genero", "nome_genero")
    .select("sk_genre_id", "nome_genero")
    .sort("nome_genero")
)

print("Quality Checks para gold.dim_genres:\n")
print(f"  dim_genres: {df_dim_genres.count()} gêneros únicos")
distinct_sk = df_dim_genres.select("sk_genre_id").distinct().count()
print(f"  sk_genre_id único: {distinct_sk == df_dim_genres.count()}")

df_dim_genres.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.dim_genres")

print(f"\ngold.dim_genres gravada")



[2/8] Criando: gold.dim_genres

Quality Checks para gold.dim_genres:

  dim_genres: 19 gêneros únicos
  sk_genre_id único: True

gold.dim_genres gravada


---

## BLOCO 3: gold.dim_people

**Origem:** silver.tb_pessoas_empresas (Ator, Diretor, Roteirista)

### Regras de Negócio Aplicadas:

#### REGRA 1: Filtrar apenas tipos de pessoa
`Motivo: Produtoras vão para dimensão separada. Esta consolida PESSOAS físicas`

#### REGRA 2: Gerar SK único por pessoa
`Motivo: Identificador estável para joins com fact tables`

#### REGRA 3: Manter tipo de pessoa
`Motivo: Permite análises por profissão (atores vs diretores vs roteiristas)`

***

In [0]:
print("\n[3/8] Criando: gold.dim_people\n")

df_silver_people = spark.table(f"{silver_schema}.tb_pessoas_empresas")

df_dim_people = (
    df_silver_people
    .filter(F.col("tipo_entidade").isin(["Ator", "Diretor", "Roteirista"]))
    .select("nome_entidade", "tipo_entidade")
    .distinct()
    .withColumn(
        "sk_person_id",
        generate_sk_udf(F.concat_ws("|", F.col("nome_entidade"), F.col("tipo_entidade")))
    )
    .withColumnRenamed("nome_entidade", "nome_pessoa")
    .withColumnRenamed("tipo_entidade", "tipo_pessoa")
    .select("sk_person_id", "nome_pessoa", "tipo_pessoa")
    .sort("nome_pessoa")
)

print("Quality Checks para gold.dim_people:\n")
total_rows = df_dim_people.count()
print(f"  dim_people: {total_rows} pessoas únicas")
distinct_sk = df_dim_people.select("sk_person_id").distinct().count()
print(f"  sk_person_id único: {distinct_sk == total_rows}")

df_dim_people.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.dim_people")

print(f"\ngold.dim_people gravada")


[3/8] Criando: gold.dim_people

Quality Checks para gold.dim_people:

  dim_people: 201770 pessoas únicas
  sk_person_id único: True

gold.dim_people gravada


---

## BLOCO 4: gold.dim_companies

**Origem:** silver.tb_pessoas_empresas (Produtora)

### Regras de Negócio Aplicadas:

#### REGRA 1: Filtrar apenas produtoras
`Motivo: Entidades corporativas, não pessoas físicas. Dimensão separada para análises por estúdio`

#### REGRA 2: Gerar SK único
`Motivo: Identificador estável para relacionamentos`

#### REGRA 3: Ordenar alfabeticamente
`Motivo: Determinismo e usabilidade em querys SQL`

***

In [0]:
print("\n[4/8] Criando: gold.dim_companies\n")

df_silver_companies = spark.table(f"{silver_schema}.tb_pessoas_empresas")

df_dim_companies = (
    df_silver_companies
    .filter(F.col("tipo_entidade") == "Produtora")
    .select("nome_entidade")
    .distinct()
    .withColumn("sk_company_id", generate_sk_udf(F.col("nome_entidade")))
    .withColumnRenamed("nome_entidade", "nome_produtora")
    .select("sk_company_id", "nome_produtora")
    .sort("nome_produtora")
)

print("Quality Checks para gold.dim_companies:\n")
print(f"  dim_companies: {df_dim_companies.count()} produtoras únicas")
distinct_sk = df_dim_companies.select("sk_company_id").distinct().count()
print(f"  sk_company_id único: {distinct_sk == df_dim_companies.count()}")

df_dim_companies.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.dim_companies")

print(f"\ngold.dim_companies gravada")



[4/8] Criando: gold.dim_companies

Quality Checks para gold.dim_companies:

  dim_companies: 46655 produtoras únicas
  sk_company_id único: True

gold.dim_companies gravada


---

## BLOCO 5: gold.dim_reviews

**Origem:** silver.tb_avaliacoes_usuarios (agregado por filme)

### Regras de Negócio Aplicadas:

#### REGRA 1: Agregar reviews por filme
`Motivo: Converter centenas de reviews por filme em métricas consolidadas (média, mín, máx, desvio padrão)`

#### REGRA 2: Gerar SK para cada filme
`Motivo: Permitir join eficiente com fact table`

#### REGRA 3: Manter id_filme natural
`Motivo: Rastreabilidade; permitir auditoria`

***

In [0]:
print("\n[5/8] Criando: gold.dim_reviews\n")

df_silver_reviews = spark.table(f"{silver_schema}.tb_avaliacoes_usuarios")
df_dim_movies = spark.table(f"{gold_schema}.dim_movies")

df_dim_reviews = (
    df_silver_reviews
    .groupBy("id_filme")
    .agg(
        F.count("*").alias("qtd_avaliacoes_usuarios"),
        F.avg("nota_usuario").alias("nota_media_usuarios"),
        F.min("nota_usuario").alias("nota_min_usuarios"),
        F.max("nota_usuario").alias("nota_max_usuarios"),
        F.stddev("nota_usuario").alias("nota_stddev_usuarios")
    )
)

df_dim_reviews = (
    df_dim_reviews
    .join(df_dim_movies.select("sk_movie_id", "id_filme"), on="id_filme", how="inner")
    .withColumn("sk_review_id", generate_sk_udf(F.col("id_filme")))
    .select(
        "sk_review_id",
        "sk_movie_id",
        "id_filme",
        "qtd_avaliacoes_usuarios",
        "nota_media_usuarios",
        "nota_min_usuarios",
        "nota_max_usuarios",
        "nota_stddev_usuarios"
    )
)

print("Quality Checks para gold.dim_reviews:\n")
print(f"  dim_reviews: {df_dim_reviews.count()} filmes com avaliações")
valid_ratings = df_dim_reviews.filter((F.col('nota_media_usuarios') >= 0) & (F.col('nota_media_usuarios') <= 10)).count()
print(f"  nota_media_usuarios entre 0 e 10: {valid_ratings == df_dim_reviews.count()}")

df_dim_reviews.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.dim_reviews")

print(f"\ngold.dim_reviews gravada")



[5/8] Criando: gold.dim_reviews

Quality Checks para gold.dim_reviews:

  dim_reviews: 26111 filmes com avaliações
  nota_media_usuarios entre 0 e 10: True

gold.dim_reviews gravada


---

## BLOCO 6: gold.fact_movies_performance

**Origem:** silver.tb_financeiro_filmes + silver.tb_metricas_engajamento (JOIN 1:1)

### Regras de Negócio Aplicadas:

#### REGRA 1: Desduplicar Silver ANTES do join
`Motivo: tb_financeiro_filmes (106K→99K único) + tb_metricas (107K→99K único). Sem desdup + join = explosão de linhas`

#### REGRA 2: INNER JOIN garante 1:1
`Motivo: Apenas filmes que têm FINANCIALS E METRICS aparecem na fact table`

#### REGRA 3: Validar integridade rigorosa
`Motivo: Uma fact table com dados duplicados distorce análises de negócio`

#### REGRA 4: Manter id_filme natural
`Motivo: Rastreabilidade cruzada com dimensões`

***

In [0]:
print("\n[6/8] Criando: gold.fact_movies_performance\n")

df_silver_info = spark.table(f"{silver_schema}.tb_info_filmes")
df_silver_financials = spark.table(f"{silver_schema}.tb_financeiro_filmes")
df_silver_metrics = spark.table(f"{silver_schema}.tb_metricas_engajamento")
df_dim_movies = spark.table(f"{gold_schema}.dim_movies")

# Desduplicar Silver ANTES do join
df_silver_financials_dedup = df_silver_financials.dropDuplicates(["id_filme"])
df_silver_metrics_dedup = df_silver_metrics.dropDuplicates(["id_filme"])

print(f"  Desduplicação:")
print(f"    - tb_financeiro_filmes: {df_silver_financials.count()} → {df_silver_financials_dedup.count()} (único)")
print(f"    - tb_metricas_engajamento: {df_silver_metrics.count()} → {df_silver_metrics_dedup.count()} (único)")

# Join (financials × metrics)
df_fact = (
    df_silver_financials_dedup
    .join(df_silver_metrics_dedup, on="id_filme", how="inner")
)

print(f"  Join (financials × metrics): {df_fact.count()} registros")

# Join com dim_movies
df_dim_movies_keys = df_dim_movies.select("sk_movie_id", "id_filme")

df_fact = (
    df_fact
    .join(df_dim_movies_keys, on="id_filme", how="inner")
)

print(f"  Join (fact × dim_movies): {df_fact.count()} registros com SK")

df_fact_movies_performance = (
    df_fact
    .select(
        "sk_movie_id",
        "id_filme",
        "orcamento_usd",
        "receita_usd",
        "lucro_usd",
        "orcamento_brl",
        "receita_brl",
        "lucro_brl",
        "margem_lucro_pct",
        "popularidade",
        "nota_media_tmdb",
        "qtd_votos_tmdb",
        "nota_media_imdb",
        "qtd_votos_imdb"
    )
)

# VALIDAÇÕES
print("\nQuality Checks para gold.fact_movies_performance:\n")
count_total = df_fact_movies_performance.count()
count_sk_unique = df_fact_movies_performance.select("sk_movie_id").distinct().count()
count_id_unique = df_fact_movies_performance.select("id_filme").distinct().count()

print(f"  Total de registros: {count_total:,}")
print(f"  SK's únicos: {count_sk_unique:,}")
print(f"  ID's únicos: {count_id_unique:,}")

if count_sk_unique == count_total == count_id_unique:
    print(f"    └─ VALIDACAO PASSOU: 1 registro por filme (1:1)")
else:
    error_msg = []
    if count_sk_unique != count_total:
        error_msg.append(f"SK's duplicadas ({count_total} vs {count_sk_unique})")
    if count_id_unique != count_total:
        error_msg.append(f"ID's duplicados ({count_total} vs {count_id_unique})")
    print(f"    └─ FALHO: {', '.join(error_msg)}")
    raise Exception(f"Fact table incorreta: {count_total} total vs {count_sk_unique} SK's vs {count_id_unique} IDs")

df_fact_movies_performance.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.fact_movies_performance")

print(f"\ngold.fact_movies_performance gravada com sucesso")



[6/8] Criando: gold.fact_movies_performance

  Desduplicação:
    - tb_financeiro_filmes: 743155 → 99006 (único)
    - tb_metricas_engajamento: 751548 → 99013 (único)
  Join (financials × metrics): 99006 registros
  Join (fact × dim_movies): 97742 registros com SK

Quality Checks para gold.fact_movies_performance:

  Total de registros: 97,742
  SK's únicos: 97,742
  ID's únicos: 97,742
    └─ VALIDACAO PASSOU: 1 registro por filme (1:1)

gold.fact_movies_performance gravada com sucesso


---

## BLOCO 7: Bridge Tables (M:N Relationships)

**Origem:** silver.tb_generos, silver.tb_pessoas_empresas + dimensões Gold

### Regras de Negócio Aplicadas:

#### REGRA 1: Bridge para Filme ↔ Gênero
`Motivo: Um filme tem múltiplos gêneros; um gênero tem múltiplos filmes. M:N resolvido via tabela de junção`

#### REGRA 2: Bridge para Filme ↔ Pessoa
`Motivo: Um filme tem múltiplos atores/diretores/roteiristas; uma pessoa aparece em múltiplos filmes`

#### REGRA 3: Bridge para Filme ↔ Produtora
`Motivo: Um filme pode ser coproduzido por múltiplas empresas`

#### REGRA 4: Usar INNER JOIN com dimensões
`Motivo: Garantir que SK existem (integridade referencial)`

#### REGRA 5: Manter apenas SK's (chaves estrangeiras)
`Motivo: Bridge table é pura conexão; nenhum atributo necessário além dos relacionamentos`

***

In [0]:
print("\n[7/8] Criando: Bridge Tables\n")

df_silver_genres = spark.table(f"{silver_schema}.tb_generos")
df_silver_people = spark.table(f"{silver_schema}.tb_pessoas_empresas")
df_dim_movies = spark.table(f"{gold_schema}.dim_movies")
df_dim_genres = spark.table(f"{gold_schema}.dim_genres")
df_dim_people = spark.table(f"{gold_schema}.dim_people")
df_dim_companies = spark.table(f"{gold_schema}.dim_companies")

# BRIDGE 1: bridge_movie_genre
df_bridge_movie_genre = (
    df_silver_genres
    .join(df_dim_movies.select("sk_movie_id", "id_filme"), on="id_filme", how="inner")
    .join(df_dim_genres.select("sk_genre_id", "nome_genero"), on=F.col("genero") == F.col("nome_genero"), how="inner")
    .select("sk_movie_id", "sk_genre_id")
    .distinct()
)

df_bridge_movie_genre.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.bridge_movie_genre")
print(f"  bridge_movie_genre: {df_bridge_movie_genre.count()} relacionamentos")

# BRIDGE 2: bridge_movie_person
# IMPORTANTE: dim_people tem uma linha por combinação (nome_pessoa + tipo_pessoa),
# já que a mesma pessoa pode atuar em papéis diferentes (ex.: Ator em um filme, Diretor em outro).
# O join precisa casar por nome E tipo simultaneamente — caso contrário, um crédito
# de "Ator" pode casar erroneamente com a linha "Diretor" da mesma pessoa em dim_people,
# gerando relacionamentos incorretos na bridge (produto cartesiano indevido por nome).
df_bridge_movie_person = (
    df_silver_people
    .filter(F.col("tipo_entidade").isin(["Ator", "Diretor", "Roteirista"]))
    .join(df_dim_movies.select("sk_movie_id", "id_filme"), on="id_filme", how="inner")
    .join(
        df_dim_people.select("sk_person_id", "nome_pessoa", "tipo_pessoa"),
        on=(
            (F.col("nome_entidade") == F.col("nome_pessoa")) &
            (F.col("tipo_entidade") == F.col("tipo_pessoa"))
        ),
        how="inner"
    )
    .select("sk_movie_id", "sk_person_id")
    .distinct()
)

df_bridge_movie_person.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.bridge_movie_person")
print(f"  bridge_movie_person: {df_bridge_movie_person.count()} relacionamentos")

# BRIDGE 3: bridge_movie_company
df_bridge_movie_company = (
    df_silver_people
    .filter(F.col("tipo_entidade") == "Produtora")
    .join(df_dim_movies.select("sk_movie_id", "id_filme"), on="id_filme", how="inner")
    .join(df_dim_companies.select("sk_company_id", "nome_produtora"), on=F.col("nome_entidade") == F.col("nome_produtora"), how="inner")
    .select("sk_movie_id", "sk_company_id")
    .distinct()
)

df_bridge_movie_company.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.bridge_movie_company")
print(f"  bridge_movie_company: {df_bridge_movie_company.count()} relacionamentos")

print(f"\nBridge Tables gravadas")


[7/8] Criando: Bridge Tables

  bridge_movie_genre: 56004 relacionamentos
  bridge_movie_person: 263076 relacionamentos
  bridge_movie_company: 61440 relacionamentos

Bridge Tables gravadas


---

## BLOCO 8: gold.genai_movies_context (RAG Context)

**Origem:** dim_movies + fact_movies_performance + bridges + dimensões

### Regras de Negócio Aplicadas:

#### REGRA 1: Base context com INNER JOIN
`Motivo: Garantir 1:1 com fact table (só filmes que têm dados financeiros E de engajamento)`

#### REGRA 2: Agregar gêneros em lista
`Motivo: Um filme tem múltiplos gêneros; consolidar em coluna STRING para RAG`

#### REGRA 3: Agregar pessoas com tipo
`Motivo: Incluir informação de profissão (Ator, Diretor, etc) no contexto`

#### REGRA 4: LEFT JOIN para enriquecimento
`Motivo: Nem todo filme tem gênero/pessoa; preservar contexto mesmo com dados parciais`

#### REGRA 5: Construir documento LLM-friendly
`Motivo: Formato narrativo legível para modelos de linguagem (RAG/GenAI)`

#### REGRA 6: Usar casting correto (string literal, não StringType())
`Motivo: PySpark .cast() requer literais string, não objetos de tipo`

#### REGRA 7: Desduplicar resultado final
`Motivo: LEFT JOINs com agregações podem criar duplicatas`

***

In [0]:
print("\n[8/8] Criando: gold.genai_movies_context\n")

df_dim_movies = spark.table(f"{gold_schema}.dim_movies")
df_fact = spark.table(f"{gold_schema}.fact_movies_performance")
df_bridge_genre = spark.table(f"{gold_schema}.bridge_movie_genre")
df_bridge_person = spark.table(f"{gold_schema}.bridge_movie_person")
df_dim_genres = spark.table(f"{gold_schema}.dim_genres")
df_dim_people = spark.table(f"{gold_schema}.dim_people")

# Base context (INNER JOIN garante 1:1)
df_context = (
    df_dim_movies
    .select("sk_movie_id", "id_filme", "titulo", "ano_lancamento", "sinopse")
    .join(df_fact.select("sk_movie_id", "receita_brl", "orcamento_brl"), on="sk_movie_id", how="inner")
)

print(f"  Base context: {df_context.count()} registros (1:1 com fact_table)")

# Agregar gêneros
df_genres_agg = (
    df_bridge_genre
    .join(df_dim_genres, on="sk_genre_id", how="inner")
    .groupBy("sk_movie_id")
    .agg(F.concat_ws(", ", F.collect_list("nome_genero")).alias("generos"))
)

# Agregar pessoas
df_people_agg = (
    df_bridge_person
    .join(df_dim_people, on="sk_person_id", how="inner")
    .groupBy("sk_movie_id")
    .agg(
        F.concat_ws(", ", F.collect_list(
            F.concat(F.col("nome_pessoa"), F.lit(" ("), F.col("tipo_pessoa"), F.lit(")"))
        )).alias("pessoas")
    )
)

# Consolidar (LEFT JOIN porque nem todo filme tem gênero/pessoa)
df_genai_context = (
    df_context
    .join(df_genres_agg, on="sk_movie_id", how="left")
    .join(df_people_agg, on="sk_movie_id", how="left")
)

print(f"  After joins: {df_genai_context.count()} registros")

# Construir documento LLM (usar "string" literal, não StringType())
df_genai_context = (
    df_genai_context
    .withColumn(
        "receita_texto",
        F.when(F.col("receita_brl").isNotNull(),
               F.cast(F.round(F.col("receita_brl") / 1_000_000, 2), "string"))
        .otherwise(F.lit("[informação não disponível]"))
    )
    .withColumn(
        "orcamento_texto",
        F.when(F.col("orcamento_brl").isNotNull(),
               F.cast(F.round(F.col("orcamento_brl") / 1_000_000, 2), "string"))
        .otherwise(F.lit("[informação não disponível]"))
    )
    .withColumn("llm_context_document",
        F.concat(
            F.lit("O filme "), F.col("titulo"), F.lit(", lançado no ano de "), 
            F.col("ano_lancamento"), F.lit(", faturou R$ "), 
            F.col("receita_texto"),
            F.lit(" milhões e teve um custo de R$ "),
            F.col("orcamento_texto"),
            F.lit(" milhões. "),
            F.lit("Estrelado por "), 
            F.coalesce(F.col("pessoas"), F.lit("elenco desconhecido")), 
            F.lit(", o filme possui a seguinte sinopse: "), 
            F.coalesce(F.col("sinopse"), F.lit("Sinopse não disponível")),
            F.lit(". Os gêneros associados são: "), 
            F.coalesce(F.col("generos"), F.lit("gêneros desconhecidos")), F.lit(".")
        )
    )
    .select("id_filme", "titulo", "llm_context_document")
    .dropDuplicates(["id_filme"])
)

# VALIDAÇÕES
print("\nQuality Checks para gold.genai_movies_context:\n")
count_total = df_genai_context.count()
count_unique = df_genai_context.select("id_filme").distinct().count()

print(f"  Total de registros: {count_total:,}")
print(f"  ID's únicos: {count_unique:,}")

if count_unique == count_total:
    print(f"    └─ VALIDACAO PASSOU: 1 contexto por filme")
else:
    print(f"    └─ AVISO: {count_total - count_unique:,} registros duplicados")

df_genai_context.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.genai_movies_context")

print(f"\ngold.genai_movies_context gravada com sucesso")



[8/8] Criando: gold.genai_movies_context

  Base context: 97742 registros (1:1 com fact_table)
  After joins: 97742 registros

Quality Checks para gold.genai_movies_context:

  Total de registros: 97,742
  ID's únicos: 97,742
    └─ VALIDACAO PASSOU: 1 contexto por filme

gold.genai_movies_context gravada com sucesso


In [0]:
print("\n=== Desafio de Analytics ===\n")

print("1. Receita total em R$ de todos os filmes\n")

df_q1 = (
    spark.table(f"{gold_schema}.fact_movies_performance")
    .agg(F.round(F.sum("receita_brl"), 2).alias("receita_total_brl"))
)

display(df_q1)

print("2. Top 5 filmes por popularidade\n")

df_q2 = (
    spark.table(f"{gold_schema}.fact_movies_performance").alias("f")
    .join(spark.table(f"{gold_schema}.dim_movies").alias("m"), "sk_movie_id")
    .select(F.col("m.titulo"), F.col("f.popularidade"))
    .orderBy(F.col("popularidade").desc())
    .limit(5)
)

display(df_q2)

print("3. Quantidade de filmes por gênero\n")

df_q3 = (
    spark.table(f"{gold_schema}.bridge_movie_genre").alias("b")
    .join(spark.table(f"{gold_schema}.dim_genres").alias("g"), "sk_genre_id")
    .groupBy("g.nome_genero")
    .agg(F.count("b.sk_movie_id").alias("qtd_filmes"))
    .orderBy(F.col("qtd_filmes").desc())
)

display(df_q3)

from pyspark.sql.window import Window

print("4. Top 10 filmes por receita com ranking\n")

window_receita = Window.orderBy(F.col("f.receita_usd").desc())

df_q4 = (
    spark.table(f"{gold_schema}.fact_movies_performance").alias("f")
    .join(spark.table(f"{gold_schema}.dim_movies").alias("m"), "sk_movie_id")
    .select(
        F.col("m.titulo"),
        F.col("f.receita_usd"),
        F.col("f.receita_brl"),
        F.rank().over(window_receita).alias("ranking")
    )
    .orderBy("ranking")
    .limit(10)
)

display(df_q4)



print("5. Ator com mais participações nos últimos 2 anos\n")

data_limite_2anos = (
    spark.table(f"{gold_schema}.dim_movies")
    .filter(F.col("data_lancamento") <= F.current_date())
    .agg(F.max("data_lancamento"))
    .collect()[0][0]
)
data_corte_2anos = F.date_sub(F.lit(data_limite_2anos), 365 * 2)

df_q5 = (
    spark.table(f"{gold_schema}.dim_movies").alias("m")
    .filter(
        (F.col("m.data_lancamento") <= F.lit(data_limite_2anos)) &
        (F.col("m.data_lancamento") >= data_corte_2anos)
    )
    .join(spark.table(f"{gold_schema}.bridge_movie_person").alias("b"), "sk_movie_id")
    .join(spark.table(f"{gold_schema}.dim_people").alias("p"), "sk_person_id")
    .filter(F.col("p.tipo_pessoa") == "Ator")
    .groupBy("p.nome_pessoa")
    .agg(F.count("m.sk_movie_id").alias("qtd_participacoes"))
    .orderBy(F.col("qtd_participacoes").desc())
    .limit(1)
)

display(df_q5)

print("6. Produtora com maior lucro nos últimos 5 anos\n")

data_limite_5anos = data_limite_2anos  # mesma data-base (data máxima de lançamento realizado)
data_corte_5anos = F.date_sub(F.lit(data_limite_5anos), 365 * 5)

df_q6 = (
    spark.table(f"{gold_schema}.dim_movies").alias("m")
    .filter(
        (F.col("m.data_lancamento") <= F.lit(data_limite_5anos)) &
        (F.col("m.data_lancamento") >= data_corte_5anos)
    )
    .join(spark.table(f"{gold_schema}.fact_movies_performance").alias("f"), "sk_movie_id")
    .join(spark.table(f"{gold_schema}.bridge_movie_company").alias("b"), "sk_movie_id")
    .join(spark.table(f"{gold_schema}.dim_companies").alias("c"), "sk_company_id")
    .filter(F.col("f.lucro_usd").isNotNull())
    .groupBy("c.nome_produtora")
    .agg(F.round(F.sum("f.lucro_usd"), 2).alias("lucro_total_usd"))
    .orderBy(F.col("lucro_total_usd").desc())
    .limit(1)
)

display(df_q6)


=== Desafio de Analytics ===

1. Receita total em R$ de todos os filmes



receita_total_brl
8.5706160230584E11


2. Top 5 filmes por popularidade



titulo,popularidade
Nightwish: Decades (live In Buenos Aires),2.018123410567891E38
Love: Augmented,1.0252800328164457E21
Silent Displacement – The Unknown Deportation Of Ingria,6.300019465E12
Asylum Of The Devil,2.0093201020121E12
Konrad Mägi,2.019202E7


3. Quantidade de filmes por gênero



...

titulo,receita_usd,receita_brl,ranking
Avengers: Endgame,2800000000.00,1.431108E10,1
Avatar: The Way Of Water,2320250281.00,1.185903121122E10,2
Avengers: Infinity War,2052415039.00,1.049009850583E10,3
Spider-man: No Way Home,1921847111.00,9.82275276903E9,4
The Lion King,1663075401.00,8.50014468205E9,5
Top Gun: Maverick,1488732821.00,7.60906232141E9,6
Barbie,1428545028.00,7.30143649261E9,7
The Super Mario Bros. Movie,1355725263.00,6.92924739172E9,8
Black Panther,1349926083.00,6.89960720282E9,9
Star Wars: The Last Jedi,1332698830.00,6.81155699001E9,10


5. Ator com mais participações nos últimos 2 anos



nome_pessoa,qtd_participacoes
"Kevin Hart, Nathalie Emmanuel, John Cena, Ben Schwartz, Paula Pell, Josh Hartnett, Milana Vayntrub, John Travolta",57


6. Produtora com maior lucro nos últimos 5 anos



nome_produtora,lucro_total_usd
"Marvel Studios, Kevin Feige Productions",2138205367.00
